In [6]:
import os

os.environ["RAY_EXPERIMENTAL_NOSET_CUDA_VISIBLE_DEVICES"] = "1"
os.environ["TRAIN_ENABLE_SHARE_CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [9]:
import ray
from ray.train import ScalingConfig
from torchvision.models import resnet18
import ray.train.torch
import torch
import time
import os

ray.init()


def trainpred_func2(config):
    print(f"{os.environ['RAY_EXPERIMENTAL_NOSET_CUDA_VISIBLE_DEVICES']=}")
    print(f"{os.environ['CUDA_VISIBLE_DEVICES']=}")
    model = resnet18(num_classes=10)
   
    cuda_dev=torch.device('cuda',ray.train.get_context().get_local_rank())
    model=ray.train.torch.prepare_model(model,cuda_dev)
    time.sleep(10)


scaling_config = ray.train.ScalingConfig(num_workers=2, use_gpu=True, resources_per_worker={"GPU":.1})
trainer = ray.train.torch.TorchTrainer(trainpred_func2,scaling_config=scaling_config)
trainer.fit()    

2026-06-22 08:38:33,950	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
(TrainController pid=373515) Requesting resources: {'GPU': 0.1} * 2
(TrainController pid=373515) Attempting to start training worker group of size 2 with the following resources: [{'GPU': 0.1}] * 2
(RayTrainWorker pid=373687) Setting up process group for: env:// [rank=0, world_size=2]
(RayTrainWorker pid=373688) os.environ['RAY_EXPERIMENTAL_NOSET_CUDA_VISIBLE_DEVICES']='1'
(RayTrainWorker pid=373688) os.environ['CUDA_VISIBLE_DEVICES']='0,1'
(TrainController pid=373515) Started training worker group of size 2: 
(TrainController pid=373515) - (ip=172.21.0.3, pid=373687) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=373515) - (ip=172.21.0.3, pid=373688) world_rank=1, local_rank=1, node_rank=0
(RayTrainWorker pid=373687) os.environ['RAY_EXPERIMENTAL_NOSET_CUDA_VISIBLE_DEVICES']='1'
(RayTrainWorker pid=373687) os.environ['CUDA_VISIBLE_DEVICES']='0,1'
(RayT

Result(metrics=None, checkpoint=None, error=None, path='/home/ray/ray_results/ray_train_run-2026-06-22_08-38-34', metrics_dataframe=None, best_checkpoints=[], _storage_filesystem=<pyarrow._fs.LocalFileSystem object at 0x7ec7022edfb0>)

In [11]:
ray.shutdown()